In [1]:
import os
data_dir = '/home/devang/Projects/PilotCrew/Data-Science-Bench/tasks/task_10/data'
print(os.listdir(data_dir))

['depots.csv', 'shipments.csv', 'scans.csv']


In [2]:
import pandas as pd
shipments = pd.read_csv(os.path.join(data_dir, 'shipments.csv'))
depots = pd.read_csv(os.path.join(data_dir, 'depots.csv'))
scans = pd.read_csv(os.path.join(data_dir, 'scans.csv'))

print("Shipments:")
print(shipments.head())
print("\nDepots:")
print(depots.head())
print("\nScans:")
print(scans.head())

Shipments:
  shipment_id   ship_date origin_depot dest_depot carrier  weight_kg  \
0     SHP0001  2024-08-01          D03        D05     CR2         12   
1     SHP0002  2024-08-02          D05        D02     CR3         19   
2     SHP0003  2024-08-03          D01        D05     CR1         26   
3     SHP0004  2024-08-04          D03        D02     CR2         33   
4     SHP0005  2024-08-05          D05        D06     CR3          9   

   promised_days  
0              3  
1              1  
2              3  
3              1  
4              3  

Depots:
  depot_id   region
0      D01     West
1      D02     East
2      D03    South
3      D04    North
4      D05  Central

Scans:
  shipment_id            scan_time        status
0     SHP0001  2024-08-01 09:00:00     picked_up
1     SHP0001  2024-08-01 16:00:00  hub_received
2     SHP0001  2024-08-02 06:00:00    in_transit
3     SHP0001  2024-08-02 19:00:00    in_transit
4     SHP0001  2024-08-03 03:00:00     delivered


In [3]:
# Calculate median weight per origin depot
median_weights = shipments.groupby('origin_depot')['weight_kg'].median().reset_index()
median_weights.rename(columns={'weight_kg': 'median_weight'}, inplace=True)

# Merge back to shipments
shipments_with_median = pd.merge(shipments, median_weights, on='origin_depot')

# Filter shipments heavier than depot's median
heavy_shipments = shipments_with_median[shipments_with_median['weight_kg'] > shipments_with_median['median_weight']].copy()

print(f"Total shipments: {len(shipments)}")
print(f"Heavy shipments: {len(heavy_shipments)}")

# Get delivery times
delivered_scans = scans[scans['status'] == 'delivered'].copy()
delivered_scans['scan_time'] = pd.to_datetime(delivered_scans['scan_time'])
delivered_scans['delivery_date'] = pd.to_datetime(delivered_scans['scan_time'].dt.date)

# Merge delivery times with heavy shipments
heavy_shipments['ship_date'] = pd.to_datetime(heavy_shipments['ship_date'])
heavy_shipments_del = pd.merge(heavy_shipments, delivered_scans[['shipment_id', 'delivery_date']], on='shipment_id', how='left')

# Calculate actual days taken
heavy_shipments_del['actual_days'] = (heavy_shipments_del['delivery_date'] - heavy_shipments_del['ship_date']).dt.days

print(heavy_shipments_del[['shipment_id', 'ship_date', 'delivery_date', 'actual_days', 'promised_days']].head())


Total shipments: 60
Heavy shipments: 30
  shipment_id  ship_date delivery_date  actual_days  promised_days
0     SHP0003 2024-08-03    2024-08-04            1              3
1     SHP0004 2024-08-04    2024-08-06            2              1
2     SHP0007 2024-08-07    2024-08-09            2              3
3     SHP0008 2024-08-08    2024-08-10            2              1
4     SHP0012 2024-08-12    2024-08-14            2              1


In [4]:
# Calculate percentage exceeding promised delivery window
exceeded = heavy_shipments_del[heavy_shipments_del['actual_days'] > heavy_shipments_del['promised_days']]
percentage = (len(exceeded) / len(heavy_shipments_del)) * 100
print(f"Percentage exceeded: {round(percentage, 1)}")

Percentage exceeded: 36.7
